In [ ]:
# !pip install yfinance
# !pip install TA-Lib
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy

In [ ]:
# Clone repo for lib/ access (Colab only)
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
import sys, os
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
from itertools import product

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)
from lib.data_manager import download, download_multi, setup_colab_secrets

In [ ]:
# Data: yfinance (daily) | Alpaca (intraday) — auto-selected by data_manager
# For Colab: run setup_colab_secrets() if using Alpaca intraday features
# EMA CROSSOVER v2 - HIGH-FREQUENCY TRADE GENERATION
# Improvements: trailing ATR stop, pullback re-entry, ADX filter, multi-asset

TICKERS = ['QQQ', 'SPY', 'NVDA', 'AMD', 'MSFT', 'AMZN', 'META', 'AAPL']
PRIMARY_TICKER = 'QQQ'
START_DATE = '2018-01-01'

all_data = download_multi(TICKERS, START_DATE)
for t, df in all_data.items():
    print(f"  {t}: {len(df)} bars")

stock_data = all_data[PRIMARY_TICKER]
TICKER = PRIMARY_TICKER
print(f"\nPrimary: {TICKER} ({len(stock_data)} bars)")

In [ ]:
# TECHNICAL ANALYSIS INDICATORS
close_arr = stock_data['Close'].values.astype(float)
high_arr  = stock_data['High'].values.astype(float)
low_arr   = stock_data['Low'].values.astype(float)
volume_arr = stock_data['Volume'].values.astype(float)
idx = stock_data.index

SMA_20 = talib.SMA(close_arr, 20); SMA_50 = talib.SMA(close_arr, 50)
EMA_12 = talib.EMA(close_arr, 12); EMA_26 = talib.EMA(close_arr, 26)
RSI_14 = talib.RSI(close_arr, 14)
ATR_14 = talib.ATR(high_arr, low_arr, close_arr, 14)
ADX_14 = talib.ADX(high_arr, low_arr, close_arr, 14)

indicators_df = pd.DataFrame({
    'Close': close_arr, 'SMA_20': SMA_20, 'SMA_50': SMA_50,
    'EMA_12': EMA_12, 'EMA_26': EMA_26,
    'RSI_14': RSI_14, 'ATR_14': ATR_14, 'ADX_14': ADX_14
}, index=idx)
print("Indicators computed.")
indicators_df.tail(5)

In [ ]:
# PREPARE PRICE SERIES
warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide", category=RuntimeWarning)

def get_ohlcv(df):
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy(); df.columns = [c[0] for c in df.columns]
    return (df['Close'].astype(float), df['High'].astype(float),
            df['Low'].astype(float), df['Volume'].astype(float))

close_s = stock_data['Close'].astype(float).squeeze(); close_s.name = 'price'
high_s  = stock_data['High'].astype(float).squeeze()
low_s   = stock_data['Low'].astype(float).squeeze()
vol_s   = stock_data['Volume'].astype(float).squeeze()

TRAIN_RATIO = 0.60
split_idx = int(len(close_s) * TRAIN_RATIO)
train_close = close_s.iloc[:split_idx].copy()
val_close   = close_s.iloc[split_idx:].copy()
train_high  = high_s.iloc[:split_idx].copy()
train_low   = low_s.iloc[:split_idx].copy()
train_vol   = vol_s.iloc[:split_idx].copy()

print(f"Train: {train_close.index[0].date()} to {train_close.index[-1].date()} ({len(train_close)} bars)")
print(f"Val:   {val_close.index[0].date()} to {val_close.index[-1].date()} ({len(val_close)} bars)")

# EMA Crossover v2 - High-Frequency Trade Generation

## What's New vs v1
| Feature | v1 | v2 |
|---|---|---|
| **Exit** | Wait for EMA cross-back | Trailing ATR stop (faster exits) |
| **Re-entry** | Only on new crossover | Pullback to fast EMA during uptrend |
| **Filter** | SMA trend filter | ADX >= 20 trend confirmation |
| **Assets** | Single ticker | 8-ticker scanning |

## Signal Logic (v2)
1. **Entry (crossover)**: Fast EMA crosses above Slow EMA AND ADX >= threshold
2. **Entry (pullback)**: In uptrend (fast > slow), price pulls back near fast EMA then bounces
3. **Exit**: Trailing stop at trail_atr * ATR below highest close since entry
4. All signals use previous bar indicators (anti-lookahead)

In [ ]:
# v2 SIGNAL ENGINE + PARAMETER RANGES

TRAIL_ATR_MULT = 2.0; ADX_MIN = 20; PULLBACK_ENABLED = True; PULLBACK_ATR_MULT = 0.5

def generate_v2_signals(close, high, low, volume, fast_ema, slow_ema,
                        trail_atr_mult=TRAIL_ATR_MULT, adx_min=ADX_MIN,
                        pullback=PULLBACK_ENABLED, pullback_atr_mult=PULLBACK_ATR_MULT,
                        atr_period=14):
    c = close.values.astype(float); h = high.values.astype(float)
    l = low.values.astype(float); v = volume.values.astype(float)
    idx = close.index; n = len(c)

    fast_vals = talib.EMA(c, timeperiod=fast_ema)
    slow_vals = talib.EMA(c, timeperiod=slow_ema)
    atr_vals  = talib.ATR(h, l, c, timeperiod=atr_period)
    adx_vals  = talib.ADX(h, l, c, timeperiod=atr_period)

    entries = np.zeros(n, dtype=bool); exits = np.zeros(n, dtype=bool)
    in_trade = False; highest = 0.0; stop = 0.0

    for i in range(max(slow_ema, atr_period) + 2, n):
        if np.isnan(fast_vals[i]) or np.isnan(slow_vals[i]) or np.isnan(atr_vals[i]):
            continue
        pf = fast_vals[i-1]; ps = slow_vals[i-1]; pa = atr_vals[i-1]
        padx = adx_vals[i-1] if not np.isnan(adx_vals[i-1]) else 0
        pc = c[i-1]; cc_ = c[i]
        p2f = fast_vals[i-2] if i >= 2 else np.nan
        p2s = slow_vals[i-2] if i >= 2 else np.nan

        if in_trade:
            if cc_ > highest:
                highest = cc_; stop = highest - trail_atr_mult * pa
            if cc_ < stop:
                exits[i] = True; in_trade = False; continue
            if not np.isnan(p2f) and p2f >= p2s and pf < ps:
                exits[i] = True; in_trade = False; continue
        else:
            if not np.isnan(p2f) and p2f <= p2s and pf > ps and padx >= adx_min:
                entries[i] = True; in_trade = True
                highest = cc_; stop = cc_ - trail_atr_mult * pa; continue
            if pullback and pf > ps and padx >= adx_min:
                dist = abs(pc - pf)
                if dist <= pullback_atr_mult * pa and cc_ > pc:
                    entries[i] = True; in_trade = True
                    highest = cc_; stop = cc_ - trail_atr_mult * pa; continue

    return pd.Series(entries, index=idx, dtype=bool), pd.Series(exits, index=idx, dtype=bool)


def generate_v1_signals(close, fast_ema, slow_ema, trend_filter):
    c = close.values.astype(float); idx = close.index
    fv = pd.Series(talib.EMA(c, timeperiod=fast_ema), index=idx)
    sv = pd.Series(talib.EMA(c, timeperiod=slow_ema), index=idx)
    tv = pd.Series(talib.SMA(c, timeperiod=trend_filter), index=idx)
    cs = pd.Series(c, index=idx)
    e_raw = ((fv.shift(1) <= sv.shift(1)) & (fv > sv) & (cs > tv))
    x_raw = ((fv.shift(1) >= sv.shift(1)) & (fv < sv))
    entries = pd.Series(np.where(e_raw.shift(1).isna(), False, e_raw.shift(1)), index=idx, dtype=bool)
    exits   = pd.Series(np.where(x_raw.shift(1).isna(), False, x_raw.shift(1)), index=idx, dtype=bool)
    return entries, exits

# Parameter Ranges
fast_ema_range  = list(range(5, 31, 1))    # 26 values
slow_ema_range  = list(range(20, 81, 2))   # 31 values
trail_atr_range = [1.5, 2.0, 2.5, 3.0]    # 4 values
adx_min_range   = [15, 20, 25]             # 3 values

ema_v2_combinations = []
for fast, slow, trail, adx in product(fast_ema_range, slow_ema_range, trail_atr_range, adx_min_range):
    if slow > fast + 5:
        ema_v2_combinations.append((fast, slow, trail, adx))

print(f"Fast EMA: {fast_ema_range[0]}-{fast_ema_range[-1]} ({len(fast_ema_range)})")
print(f"Slow EMA: {slow_ema_range[0]}-{slow_ema_range[-1]} ({len(slow_ema_range)})")
print(f"Trail ATR: {trail_atr_range}")
print(f"ADX min: {adx_min_range}")
print(f"\nTotal combos: {len(ema_v2_combinations)}")
print(f"\nFirst 10:")
for i, (f, s, t, a) in enumerate(ema_v2_combinations[:10], 1):
    print(f"  {i:2d}. Fast={f:2d} Slow={s:2d} Trail={t:.1f} ADX>={a}")

In [ ]:
# Initialize Results Collection
grid_search_results = []
print(f"EMA v2 Results Initialized | Testing {len(ema_v2_combinations)} combos")

In [ ]:
# EMA CROSSOVER v2 GRID SEARCH - TRAINING DATA

print("INITIATING EMA CROSSOVER v2 GRID SEARCH")
print("=" * 70)
print(f"Training: {train_close.index[0].date()} to {train_close.index[-1].date()}")
print(f"Capital: $100,000 | Fees: 0.05% | Slippage: 0.05%")
print("=" * 70)

total_combos = len(ema_v2_combinations)
grid_search_results = []; successful = 0; failed = 0; skipped = 0
years = max((train_close.index[-1] - train_close.index[0]).days / 365.25, 1e-9)

for combo_num, (fast, slow, trail, adx) in enumerate(ema_v2_combinations, 1):
    try:
        entries, exits = generate_v2_signals(train_close, train_high, train_low, train_vol,
                                             fast_ema=fast, slow_ema=slow,
                                             trail_atr_mult=trail, adx_min=adx)
        pf = vbt.Portfolio.from_signals(close=train_close, entries=entries, exits=exits,
                                        init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        trades = pf.trades; total_trades = len(trades)
        if total_trades < 5:
            skipped += 1
            continue

        trades_per_year = total_trades / years
        total_return = float(pf.total_return())
        ann_ret = float(pf.annualized_return(freq='D'))
        max_dd = float(pf.max_drawdown())
        vol = float(pf.annualized_volatility(freq='D'))
        sharpe = float(pf.sharpe_ratio(freq='D'))
        sortino = float(pf.sortino_ratio(freq='D'))
        calmar = ann_ret / abs(max_dd) if abs(max_dd) > 1e-9 else np.nan

        wr = np.nan; pf_ratio = np.nan; exp = 0.0
        aw = 0.0; al = 0.0; lw = 0.0; ll = 0.0; ws = np.nan; ls = np.nan; sqn = np.nan

        tr = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
        if tr.size > 0:
            pos = tr[tr > 0]; neg = tr[tr < 0]
            wr = (len(pos)/len(tr))*100 if len(tr) else np.nan
            g = pos.sum() if len(pos) else 0; l_ = abs(neg.sum()) if len(neg) else 0
            pf_ratio = g/l_ if l_ > 0 else np.inf
            exp = float(tr.mean())
            aw = float(pos.mean()) if len(pos) else 0
            al = float(abs(neg.mean())) if len(neg) else 0
            lw = float(pos.max()) if len(pos) else 0
            ll = float(neg.min()) if len(neg) else 0
            sqn = float(tr.mean()/tr.std()*np.sqrt(len(tr))) if tr.std() > 0 else np.nan
            try:
                ws = int(trades.winning_streak()); ls = int(trades.losing_streak())
            except:
                pass

        rets = pf.returns(); cum = (1+rets).cumprod(); pk = cum.cummax(); dd = (cum-pk)/pk
        ui = float(np.sqrt((dd.pow(2)).mean())) if len(dd) else np.nan
        pr = (aw/al) if al > 0 else np.inf
        rv = rets.values
        om = float(rv[rv>0].sum()/abs(rv[rv<0].sum())) if abs(rv[rv<0].sum()) > 0 else np.inf
        rf = total_return/abs(max_dd) if abs(max_dd) > 1e-9 else np.nan
        gp = float(rv.sum()/abs(rv[rv<0].sum())) if abs(rv[rv<0].sum()) > 0 else np.inf
        si = rf/ui if ui and ui > 0 else np.nan

        grid_search_results.append({
            "fast_ema": fast, "slow_ema": slow, "trail_atr_mult": trail, "adx_min": adx,
            "total_return": total_return, "annualized_return": ann_ret,
            "sharpe_ratio": sharpe, "sortino_ratio": sortino, "calmar_ratio": calmar,
            "omega_ratio": om, "max_drawdown": max_dd, "volatility": vol,
            "ulcer_index": ui, "win_rate": wr, "total_trades": total_trades,
            "trades_per_year": trades_per_year, "expectancy": exp,
            "profit_factor": pf_ratio, "sqn": sqn, "payoff_ratio": pr,
            "largest_win": lw, "largest_loss": ll, "avg_win_amount": aw, "avg_loss_amount": al,
            "winning_streak": ws, "losing_streak": ls,
            "recovery_factor": rf, "gain_to_pain_ratio": gp, "serenity_index": si
        })
        successful += 1
    except:
        failed += 1

    if combo_num % 500 == 0:
        print(f"Progress: {combo_num}/{total_combos} ({combo_num/total_combos*100:.1f}%) | Done: {successful} | Skip: {skipped}")

print("\n" + "=" * 70)
print("GRID SEARCH COMPLETE")
print(f"Tested: {total_combos} | OK: {successful} | Skip: {skipped} | Fail: {failed}")

if successful > 0:
    results_df = pd.DataFrame(grid_search_results)
    top_10 = results_df.nlargest(10, 'sharpe_ratio')
    print("\nTOP 10 BY SHARPE:")
    for rank, (_, row) in enumerate(top_10.iterrows(), 1):
        print(f"  #{rank} F={int(row['fast_ema'])} S={int(row['slow_ema'])} "
              f"T={row['trail_atr_mult']:.1f} ADX>={int(row['adx_min'])} | "
              f"SR={row['sharpe_ratio']:.3f} Ret={row['total_return']:.2%} "
              f"DD={row['max_drawdown']:.2%} Trades={int(row['total_trades'])} ({row['trades_per_year']:.1f}/yr)")

In [ ]:
# TOP 5 OUT-OF-SAMPLE VALIDATION
results_df = pd.DataFrame(grid_search_results)
if results_df.empty:
    print("No results.")
else:
    print("=" * 90)
    print("TOP 5 - OUT-OF-SAMPLE VALIDATION")
    print("=" * 90)
    val_high = high_s.iloc[split_idx:]; val_low = low_s.iloc[split_idx:]; val_vol = vol_s.iloc[split_idx:]
    top_5 = results_df.nlargest(5, 'sharpe_ratio'); oos_results = []
    for _, st in top_5.iterrows():
        fe=int(st['fast_ema']); se=int(st['slow_ema']); tr_=float(st['trail_atr_mult']); ad=int(st['adx_min'])
        e, x = generate_v2_signals(val_close, val_high, val_low, val_vol, fe, se, tr_, ad)
        pf = vbt.Portfolio.from_signals(close=val_close, entries=e, exits=x,
                                        init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        os_sr = float(pf.sharpe_ratio(freq='D')); os_ret = float(pf.total_return())
        os_dd = float(pf.max_drawdown()); nt = len(pf.trades)
        os_wr = np.nan; os_pf = np.nan
        if nt > 0:
            tr = pf.trades.returns.values if hasattr(pf.trades.returns,'values') else np.array(pf.trades.returns)
            if tr.size:
                p=tr[tr>0]; n=tr[tr<0]
                os_wr=(len(p)/len(tr))*100
                os_pf=p.sum()/abs(n.sum()) if len(n) and abs(n.sum())>0 else np.inf
        vy = max((val_close.index[-1]-val_close.index[0]).days/365.25, 1e-9)
        oos_results.append({'Rank':len(oos_results)+1,'Fast':fe,'Slow':se,'Trail':tr_,'ADX':ad,
            'IS_Sharpe':float(st['sharpe_ratio']),'IS_Return':float(st['total_return']),
            'IS_Trades':int(st['total_trades']),'OOS_Sharpe':os_sr,'OOS_Return':os_ret,
            'OOS_MaxDD':os_dd,'OOS_Trades':nt,'OOS_WR':os_wr,'OOS_PF':os_pf,'OOS_T/Yr':nt/vy,
            'Degrad':((os_sr-st['sharpe_ratio'])/abs(st['sharpe_ratio'])*100) if st['sharpe_ratio']!=0 else np.nan})
    print(pd.DataFrame(oos_results).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

    fig, axes = plt.subplots(1, min(5,len(oos_results)), figsize=(20,5), squeeze=False)
    for i, row in enumerate(oos_results):
        ax = axes[0][i]
        e, x = generate_v2_signals(val_close, val_high, val_low, val_vol, row['Fast'], row['Slow'], row['Trail'], row['ADX'])
        pf = vbt.Portfolio.from_signals(close=val_close, entries=e, exits=x, init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        pf.value().plot(ax=ax, color='#2563EB')
        ax.set_title(f"#{row['Rank']} F={row['Fast']} S={row['Slow']}\nSR={row['OOS_Sharpe']:.2f} T={row['OOS_Trades']}", fontsize=9)
        ax.axhline(100_000, ls='--', color='gray', alpha=0.5)
    fig.suptitle(f"OOS Equity - {TICKER} EMA v2", fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

# Parameter Sensitivity Analysis
Vary each param around best value, hold others fixed. Color: green=better, red=worse vs base.

In [ ]:
# SENSITIVITY ANALYSIS
if results_df.empty:
    print("No results.")
else:
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    fb=int(best['fast_ema']); sb=int(best['slow_ema']); tb=float(best['trail_atr_mult']); ab=int(best['adx_min'])
    print(f"Best: F={fb} S={sb} T={tb:.1f} ADX>={ab} | Sharpe={best['sharpe_ratio']:.3f}")

    cfgs = [('fast_ema',fb,list(range(max(3,fb-10),fb+11))),('slow_ema',sb,list(range(max(10,sb-15),sb+16,2))),
            ('trail_atr_mult',tb,[1.0,1.5,2.0,2.5,3.0,3.5]),('adx_min',ab,[10,15,20,25,30])]
    fig, axes = plt.subplots(2, 4, figsize=(20,10))
    for col, (pn, bv, vr) in enumerate(cfgs):
        for ri, (dl, dc, dh, dl2, dv) in enumerate([
            ("IS",train_close,train_high,train_low,train_vol),
            ("OOS",val_close,high_s.iloc[split_idx:],low_s.iloc[split_idx:],vol_s.iloc[split_idx:])]):
            ax = axes[ri][col]; sharpes = []
            for v in vr:
                kw = {'fast_ema':fb,'slow_ema':sb,'trail_atr_mult':tb,'adx_min':ab}; kw[pn]=v
                if kw['slow_ema']<=kw['fast_ema']+5: sharpes.append(np.nan); continue
                try:
                    e,x = generate_v2_signals(dc,dh,dl2,dv,**kw)
                    pf = vbt.Portfolio.from_signals(close=dc,entries=e,exits=x,init_cash=100_000,fees=0.0005,slippage=0.0005,freq='D')
                    sharpes.append(float(pf.sharpe_ratio(freq='D')))
                except: sharpes.append(np.nan)
            bs = best['sharpe_ratio']
            colors = ['gray' if np.isnan(s) else ('#059669' if (s-bs)/abs(bs)*100>10 else '#86efac' if (s-bs)/abs(bs)*100>=0 else '#fb923c' if (s-bs)/abs(bs)*100>=-10 else '#DC2626') for s in sharpes]
            ax.bar(range(len(vr)),sharpes,color=colors)
            ax.set_xticks(range(len(vr))); ax.set_xticklabels([str(v) for v in vr],fontsize=7,rotation=45)
            bi = next((vi for vi,v in enumerate(vr) if v==bv or (isinstance(bv,float) and abs(v-bv)<0.01)),None)
            if bi is not None: ax.axvline(bi,ls='--',color='#2563EB',lw=1.5)
            ax.set_title(f"{dl}: {pn}",fontsize=10); ax.set_ylabel("Sharpe")
    fig.suptitle(f"Sensitivity - {TICKER} EMA v2",fontsize=14,fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
# FEATURE ABLATION - v1 vs v2
best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
fe=int(best['fast_ema']); se=int(best['slow_ema']); tr_=float(best['trail_atr_mult']); ad=int(best['adx_min'])

configs = [
    ("v1 Baseline", lambda c,h,l,v: generate_v1_signals(c, fe, se, se+20)),
    ("v2 No Pullback", lambda c,h,l,v: generate_v2_signals(c,h,l,v,fe,se,tr_,ad,pullback=False)),
    ("v2 No ADX", lambda c,h,l,v: generate_v2_signals(c,h,l,v,fe,se,tr_,adx_min=0)),
    ("v2 Tight (1.5)", lambda c,h,l,v: generate_v2_signals(c,h,l,v,fe,se,1.5,ad)),
    ("v2 Wide (3.0)", lambda c,h,l,v: generate_v2_signals(c,h,l,v,fe,se,3.0,ad)),
    ("v2 FULL", lambda c,h,l,v: generate_v2_signals(c,h,l,v,fe,se,tr_,ad)),
]

print(f"ABLATION | Best: F={fe} S={se} T={tr_:.1f} ADX>={ad}")
print("=" * 90)
rows = []
for label, fn in configs:
    for sl, c, h, l, v in [("IS",train_close,train_high,train_low,train_vol),
                            ("OOS",val_close,high_s.iloc[split_idx:],low_s.iloc[split_idx:],vol_s.iloc[split_idx:])]:
        try:
            e, x = fn(c, h, l, v)
            pf = vbt.Portfolio.from_signals(close=c,entries=e,exits=x,init_cash=100_000,fees=0.0005,slippage=0.0005,freq='D')
            yr = max((c.index[-1]-c.index[0]).days/365.25,1e-9)
            rows.append({'Config':label,'Split':sl,'Sharpe':float(pf.sharpe_ratio(freq='D')),
                'Return':float(pf.total_return()),'MaxDD':float(pf.max_drawdown()),
                'Trades':len(pf.trades),'T/Yr':len(pf.trades)/yr,
                'WR':float((pf.trades.returns.values>0).mean()*100) if len(pf.trades) else np.nan})
        except:
            rows.append({'Config':label,'Split':sl,'Sharpe':np.nan,'Return':np.nan,'MaxDD':np.nan,'Trades':0,'T/Yr':0,'WR':np.nan})

adf = pd.DataFrame(rows)
for s in ['IS','OOS']:
    print(f"\n  {s}:")
    print(adf[adf['Split']==s][['Config','Sharpe','Return','MaxDD','Trades','T/Yr','WR']].to_string(index=False,float_format=lambda x:f"{x:.3f}"))

In [ ]:
# MULTI-ASSET OOS
best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
fe=int(best['fast_ema']); se=int(best['slow_ema']); tr_=float(best['trail_atr_mult']); ad=int(best['adx_min'])
print(f"Testing across {len(all_data)} tickers | F={fe} S={se} T={tr_:.1f} ADX>={ad}")

multi = []; nt = len(all_data); cols = min(4,nt); rows = (nt+cols-1)//cols
fig, axes = plt.subplots(rows, cols, figsize=(5*cols,4*rows), squeeze=False)
for ti,(tk,df) in enumerate(all_data.items()):
    c,h,l,v = get_ohlcv(df); sp = int(len(c)*TRAIN_RATIO)
    oc=c.iloc[sp:]; oh=h.iloc[sp:]; ol=l.iloc[sp:]; ov=v.iloc[sp:]
    try:
        e,x = generate_v2_signals(oc,oh,ol,ov,fe,se,tr_,ad)
        pf = vbt.Portfolio.from_signals(close=oc,entries=e,exits=x,init_cash=100_000,fees=0.0005,slippage=0.0005,freq='D')
        yr = max((oc.index[-1]-oc.index[0]).days/365.25,1e-9)
        sr=float(pf.sharpe_ratio(freq='D')); rt=float(pf.total_return()); dd=float(pf.max_drawdown()); tn=len(pf.trades)
        multi.append({'Ticker':tk,'SR':sr,'Return':rt,'MaxDD':dd,'Trades':tn,'T/Yr':tn/yr})
        ax=axes[ti//cols][ti%cols]; pf.value().plot(ax=ax,color='#2563EB')
        ax.axhline(100_000,ls='--',color='gray',alpha=0.5); ax.set_title(f"{tk} SR={sr:.2f} T={tn}",fontsize=10)
    except:
        multi.append({'Ticker':tk,'SR':np.nan,'Return':np.nan,'MaxDD':np.nan,'Trades':0,'T/Yr':0})
        axes[ti//cols][ti%cols].set_title(f"{tk} ERROR")
for j in range(nt,rows*cols): axes[j//cols][j%cols].set_visible(False)
fig.suptitle("EMA v2 Multi-Asset OOS",fontsize=14,fontweight='bold'); plt.tight_layout(); plt.show()
mdf = pd.DataFrame(multi); print(mdf.to_string(index=False,float_format=lambda x:f"{x:.3f}"))
print(f"\nAvg SR: {mdf['SR'].mean():.3f} | Avg T/Yr: {mdf['T/Yr'].mean():.1f}")

In [ ]:
# FTMO MONTE CARLO - v1 vs v2
import json as _json
best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
fe=int(best['fast_ema']); se=int(best['slow_ema']); tr_=float(best['trail_atr_mult']); ad=int(best['adx_min'])
vh=high_s.iloc[split_idx:]; vl=low_s.iloc[split_idx:]; vv=vol_s.iloc[split_idx:]

e2,x2 = generate_v2_signals(val_close,vh,vl,vv,fe,se,tr_,ad)
pf2 = vbt.Portfolio.from_signals(close=val_close,entries=e2,exits=x2,init_cash=100_000,fees=0.0005,slippage=0.0005,freq='D')
e1,x1 = generate_v1_signals(val_close,fe,se,se+20)
pf1 = vbt.Portfolio.from_signals(close=val_close,entries=e1,exits=x1,init_cash=100_000,fees=0.0005,slippage=0.0005,freq='D')

try:
    with open('../config/ftmo_rules.json','r') as f: ftmo=_json.load(f)
except: ftmo={"account_size":100000,"profit_target_pct":10,"max_daily_loss_pct":5,"max_total_loss_pct":10}

ACC=ftmo['account_size']; TGT=ACC*ftmo['profit_target_pct']/100
MXD=ACC*ftmo['max_daily_loss_pct']/100; MXT=ACC*ftmo['max_total_loss_pct']/100
DAYS=30; NSIM=10_000

def ftmo_mc(rets_s, label):
    r = rets_s.dropna().values
    if len(r)<5: print(f"  {label}: too few"); return None
    ps=0; bd=0; bt=0; to=0; fb=[]
    for _ in range(NSIM):
        s=np.random.choice(r,DAYS,True); b=ACC; blown=False
        for rv in s:
            pnl=b*rv; b+=pnl
            if pnl<-MXD: bd+=1; blown=True; break
            if ACC-b>MXT: bt+=1; blown=True; break
        fb.append(b)
        if not blown:
            if b>=ACC+TGT: ps+=1
            else: to+=1
    pr=ps/NSIM*100
    print(f"  {label}: Pass={pr:.1f}% DailyDD={bd/NSIM*100:.1f}% TotalDD={bt/NSIM*100:.1f}% Timeout={to/NSIM*100:.1f}%")
    return {'pass_rate':pr,'blown_daily':bd/NSIM*100,'blown_total':bt/NSIM*100,'balances':fb}

print(f"FTMO MC | {NSIM:,} sims x {DAYS} days | Acct=${ACC:,} Target=+${TGT:,}")
print("=" * 70)
mc1 = ftmo_mc(pf1.returns(), "v1 (cross only)")
mc2 = ftmo_mc(pf2.returns(), "v2 (trail+pullback)")

if mc1 and mc2:
    fig,(a1,a2) = plt.subplots(1,2,figsize=(14,5))
    for mc,lb,cl,ax in [(mc1,'v1','#94a3b8',a1),(mc2,'v2','#2563EB',a2)]:
        ax.hist(mc['balances'],bins=80,color=cl,alpha=0.8,edgecolor='white',lw=0.3)
        ax.axvline(ACC+TGT,color='#059669',ls='--',lw=2,label=f"Target ${ACC+TGT:,}")
        ax.axvline(ACC-MXT,color='#DC2626',ls='--',lw=2,label=f"Blown ${ACC-MXT:,}")
        ax.set_title(f"{lb} Pass={mc['pass_rate']:.1f}%",fontsize=13,fontweight='bold')
        ax.legend(fontsize=9)
    fig.suptitle(f"FTMO MC - {TICKER} EMA v1 vs v2",fontsize=14,fontweight='bold')
    plt.tight_layout(); plt.show()
    print(f"\nv2 improvement: +{mc2['pass_rate']-mc1['pass_rate']:.1f}pp")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# UNIVERSAL STRATEGY EXPORT — Data Files + PDF Tearsheet
# ════════════════════════════════════════════════════════════════════
# INSTRUCTIONS:
#   1. Paste at the END of any strategy notebook
#   2. Edit STRATEGY_NAME and PARAM_COLS below
#   3. Run — exports structured data + a comprehensive PDF report
# ════════════════════════════════════════════════════════════════════

import os, sys, json, datetime, hashlib, platform
from matplotlib.backends.backend_pdf import PdfPages

# ═══ EDIT THESE LINES ═══════════════════════════════════════
STRATEGY_NAME = "EMA_Crossover_v2"                                    # ← EDIT
PARAM_COLS    = ["fast_ema", "slow_ema", "trail_atr_mult", "adx_min"]     # ← EDIT
NOTES         = ""                                                   # ← Optional run notes
# ════════════════════════════════════════════════════════════════

INIT_CASH = 100_000
FEES      = 0.0005
SLIPPAGE  = 0.0005
FREQ      = 'D'

# ── Google Drive mount ──
EXPORT_DIR = "/content/strategy_exports"
IN_COLAB = 'google.colab' in sys.modules
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    EXPORT_DIR = "/content/drive/MyDrive/strategy_exports"
    IN_COLAB = True
    print("\u2705 Google Drive mounted")
except:
    print("\u26a0\ufe0f Local mode — exports to ./strategy_exports")

RUN_TIMESTAMP = datetime.datetime.now()
RUN_ID = RUN_TIMESTAMP.strftime("%Y%m%d_%H%M%S")

# ── Folder structure ──
STRAT_DIR   = os.path.join(EXPORT_DIR, STRATEGY_NAME, TICKER)
LATEST_DIR  = os.path.join(STRAT_DIR, "latest")
ARCHIVE_DIR = os.path.join(STRAT_DIR, "archive")
os.makedirs(LATEST_DIR, exist_ok=True)
os.makedirs(ARCHIVE_DIR, exist_ok=True)

# ════════════════════════════════════════════════════════════════
# Signal function — auto-detects strategy type
# ════════════════════════════════════════════════════════════════
def _compute_signals(price_s, params, high_s=None, low_s=None):
    idx = price_s.index; vals = price_s.values.astype(float)
    if STRATEGY_NAME.startswith("MACD"):
        ml, sl, _ = talib.MACD(vals, fastperiod=params['fast_period'], slowperiod=params['slow_period'], signalperiod=params['signal_period'])
        ms, ss = pd.Series(ml, index=idx), pd.Series(sl, index=idx)
        e_raw = (ms.shift(1) <= ss.shift(1)) & (ms > ss)
        x_raw = (ms.shift(1) >= ss.shift(1)) & (ms < ss)
    elif STRATEGY_NAME.startswith("RSI"):
        rsi_s = pd.Series(talib.RSI(vals, timeperiod=params['rsi_len']), index=idx)
        e_raw = (rsi_s.shift(1) <= params['oversold']) & (rsi_s > params['oversold'])
        x_raw = (rsi_s.shift(1) <= params['overbought']) & (rsi_s > params['overbought'])
    elif STRATEGY_NAME.startswith("EMA"):
        fv = pd.Series(talib.EMA(vals, timeperiod=params['fast_ema']), index=idx)
        sv = pd.Series(talib.EMA(vals, timeperiod=params['slow_ema']), index=idx)
        tv = pd.Series(talib.SMA(vals, timeperiod=params['trend_filter']), index=idx)
        cs = pd.Series(vals, index=idx)
        e_raw = ((fv.shift(1) <= sv.shift(1)) & (fv > sv) & (cs > tv))
        x_raw = ((fv.shift(1) >= sv.shift(1)) & (fv < sv))
    elif STRATEGY_NAME.startswith("Triple") or STRATEGY_NAME.startswith("TEMA"):
        e1 = pd.Series(vbt.MA.run(price_s, params['ema1_period'], ewm=True).ma.values.flatten(), index=idx)
        e2 = pd.Series(vbt.MA.run(price_s, params['ema2_period'], ewm=True).ma.values.flatten(), index=idx)
        e3 = pd.Series(vbt.MA.run(price_s, params['ema3_period'], ewm=True).ma.values.flatten(), index=idx)
        e_raw = (e1.vbt.crossed_above(e2) | e1.vbt.crossed_above(e3) | e2.vbt.crossed_above(e3))
        x_raw = (e1.vbt.crossed_below(e2) | e1.vbt.crossed_below(e3) | e2.vbt.crossed_below(e3))
    elif STRATEGY_NAME.startswith("Donchian") or STRATEGY_NAME.startswith("DC"):
        h_v = high_s.values.astype(float) if high_s is not None else vals
        l_v = low_s.values.astype(float) if low_s is not None else vals
        uc = pd.Series(talib.MAX(h_v, timeperiod=params['entry_period']), index=idx).shift(1)
        lc = pd.Series(talib.MIN(l_v, timeperiod=params['exit_period']), index=idx).shift(1)
        tf = pd.Series(talib.SMA(vals, timeperiod=params['filter_period']), index=idx).shift(1)
        cs = pd.Series(vals, index=idx)
        e_raw = (cs > uc) & (cs > tf); x_raw = (cs < lc)
    else:
        raise ValueError(f"Unknown strategy: {STRATEGY_NAME}")
    entries = pd.Series(np.where(e_raw.shift(1).isna(), False, e_raw.shift(1)), index=idx, dtype=bool)
    exits = pd.Series(np.where(x_raw.shift(1).isna(), False, x_raw.shift(1)), index=idx, dtype=bool)
    return entries, exits

# ════════════════════════════════════════════════════════════════
# Build portfolios
# ════════════════════════════════════════════════════════════════
results_df = pd.DataFrame(grid_search_results)
best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
best_params = {col: int(best[col]) for col in PARAM_COLS}
param_str = ", ".join([f"{k}={v}" for k, v in best_params.items()])

if isinstance(stock_data.columns, pd.MultiIndex):
    full_close = stock_data[('Close', TICKER)].astype(float).squeeze()
else:
    full_close = stock_data['Close'].astype(float).squeeze()
full_close.name = 'price'

split_idx = int(len(full_close) * 0.60)
train_close = full_close.iloc[:split_idx].copy()
val_close = full_close.iloc[split_idx:].copy()

# High/Low for Donchian
high_s, low_s = None, None
if STRATEGY_NAME.startswith("Donchian") or STRATEGY_NAME.startswith("DC"):
    if isinstance(stock_data.columns, pd.MultiIndex):
        high_s = stock_data[('High', TICKER)].astype(float).squeeze()
        low_s = stock_data[('Low', TICKER)].astype(float).squeeze()
    else:
        high_s = stock_data['High'].astype(float).squeeze()
        low_s = stock_data['Low'].astype(float).squeeze()

# Full sample
e_full, x_full = _compute_signals(full_close, best_params, high_s, low_s)
pf_full = vbt.Portfolio.from_signals(close=full_close, entries=e_full, exits=x_full,
                                      init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)
# IS
e_is, x_is = _compute_signals(train_close, best_params,
                                high_s.iloc[:split_idx] if high_s is not None else None,
                                low_s.iloc[:split_idx] if low_s is not None else None)
pf_is = vbt.Portfolio.from_signals(close=train_close, entries=e_is, exits=x_is,
                                    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)
# OOS
e_oos, x_oos = _compute_signals(val_close, best_params,
                                  high_s.iloc[split_idx:] if high_s is not None else None,
                                  low_s.iloc[split_idx:] if low_s is not None else None)
pf_oos = vbt.Portfolio.from_signals(close=val_close, entries=e_oos, exits=x_oos,
                                     init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)
# Buy & Hold
bh_e = pd.Series(False, index=full_close.index, dtype=bool); bh_e.iloc[0] = True
bh_x = pd.Series(False, index=full_close.index, dtype=bool)
pf_bh = vbt.Portfolio.from_signals(close=full_close, entries=bh_e, exits=bh_x,
                                    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)

# ── Extract metrics ──
trades_obj = pf_full.trades
tr = np.asarray(trades_obj.returns.values if hasattr(trades_obj.returns, 'values') else trades_obj.returns).ravel()
pnl = np.asarray(trades_obj.pnl.values if hasattr(trades_obj.pnl, 'values') else trades_obj.pnl).ravel()
pos, neg = tr[tr > 0], tr[tr < 0]
years_full = max((full_close.index[-1] - full_close.index[0]).days / 365.25, 1e-9)
daily_rets = pf_full.returns()

def safe(fn, default=None):
    try: return float(fn())
    except: return default

M = {  # Master metrics dict
    'total_return': safe(pf_full.total_return), 'ann_return': safe(lambda: pf_full.annualized_return(freq=FREQ)),
    'sharpe': safe(lambda: pf_full.sharpe_ratio(freq=FREQ)), 'sortino': safe(lambda: pf_full.sortino_ratio(freq=FREQ)),
    'max_dd': safe(pf_full.max_drawdown), 'volatility': safe(lambda: pf_full.annualized_volatility(freq=FREQ)),
    'calmar': safe(lambda: pf_full.annualized_return(freq=FREQ)) / abs(safe(pf_full.max_drawdown)) if abs(safe(pf_full.max_drawdown, 0)) > 1e-9 else None,
    'trades': len(trades_obj), 'trades_yr': len(trades_obj) / years_full,
    'win_rate': float(len(pos) / len(tr) * 100) if len(tr) > 0 else None,
    'pf': float(pos.sum() / abs(neg.sum())) if len(neg) > 0 and abs(neg.sum()) > 0 else None,
    'expectancy': float(tr.mean()) if len(tr) > 0 else None,
    'avg_win': float(pos.mean()) if len(pos) > 0 else None, 'avg_loss': float(neg.mean()) if len(neg) > 0 else None,
    'largest_win': float(pos.max()) if len(pos) > 0 else None, 'largest_loss': float(neg.min()) if len(neg) > 0 else None,
    'payoff': float(abs(pos.mean() / neg.mean())) if len(pos) > 0 and len(neg) > 0 else None,
    'is_sharpe': safe(lambda: pf_is.sharpe_ratio(freq=FREQ)), 'is_return': safe(pf_is.total_return),
    'is_dd': safe(pf_is.max_drawdown), 'is_trades': len(pf_is.trades),
    'oos_sharpe': safe(lambda: pf_oos.sharpe_ratio(freq=FREQ)), 'oos_return': safe(pf_oos.total_return),
    'oos_dd': safe(pf_oos.max_drawdown), 'oos_trades': len(pf_oos.trades),
    'bh_return': safe(pf_bh.total_return), 'bh_sharpe': safe(lambda: pf_bh.sharpe_ratio(freq=FREQ)),
    'bh_dd': safe(pf_bh.max_drawdown),
}

# ════════════════════════════════════════════════════════════════
# 1. SAVE STRUCTURED DATA FILES
# ════════════════════════════════════════════════════════════════
export_json = {
    "metadata": {
        "run_id": RUN_ID, "export_timestamp": RUN_TIMESTAMP.isoformat(),
        "export_date_human": RUN_TIMESTAMP.strftime("%B %d, %Y at %I:%M %p"),
        "strategy_name": STRATEGY_NAME, "strategy_family": STRATEGY_NAME.split("_")[0],
        "ticker": TICKER,
        "instrument_type": ("crypto" if "-USD" in TICKER and TICKER.replace("-USD","").isalpha()
                           else "forex" if "/" in TICKER or (len(TICKER) == 6 and TICKER.isalpha())
                           else "equity/etf"),
        "data_source": "yfinance", "data_interval": "1d", "currency": "USD",
        "start_date": str(full_close.index[0].date()), "end_date": str(full_close.index[-1].date()),
        "total_bars": len(full_close), "total_years": round(years_full, 2),
        "train_start": str(train_close.index[0].date()), "train_end": str(train_close.index[-1].date()),
        "train_bars": len(train_close), "val_start": str(val_close.index[0].date()),
        "val_end": str(val_close.index[-1].date()), "val_bars": len(val_close), "train_ratio": 0.60,
        "init_cash": INIT_CASH, "fees_pct": FEES, "slippage_pct": SLIPPAGE, "frequency": FREQ,
        "first_close": round(float(full_close.iloc[0]), 4), "last_close": round(float(full_close.iloc[-1]), 4),
        "python_version": sys.version.split()[0], "environment": "colab_pro" if IN_COLAB else "local",
        "grid_combos_tested": len(results_df), "param_columns": PARAM_COLS, "notes": NOTES,
    },
    "best_params": best_params,
    "metrics_full_sample": {k: v for k, v in M.items() if not k.startswith('is_') and not k.startswith('oos_') and not k.startswith('bh_')},
    "metrics_in_sample": {k.replace('is_',''): v for k, v in M.items() if k.startswith('is_')},
    "metrics_out_of_sample": {k.replace('oos_',''): v for k, v in M.items() if k.startswith('oos_')},
    "metrics_buy_hold": {k.replace('bh_',''): v for k, v in M.items() if k.startswith('bh_')},
    "grid_search_summary": {
        "top5": results_df.nlargest(5, 'sharpe_ratio')[PARAM_COLS + ['sharpe_ratio','total_return','max_drawdown']].to_dict('records'),
    }
}

# Save JSON
with open(os.path.join(LATEST_DIR, "summary.json"), 'w') as f:
    json.dump(export_json, f, indent=2, default=str)
with open(os.path.join(ARCHIVE_DIR, f"{RUN_ID}_summary.json"), 'w') as f:
    json.dump(export_json, f, indent=2, default=str)
print(f"\u2705 summary.json")

# Save CSVs
pd.DataFrame({'date': full_close.index.strftime('%Y-%m-%d'), 'strategy_return': daily_rets.values,
              'close': full_close.values, 'portfolio_value': pf_full.value().values
}).to_csv(os.path.join(LATEST_DIR, "daily_returns.csv"), index=False)
print(f"\u2705 daily_returns.csv")

pd.DataFrame({'trade_num': range(1, len(tr)+1), 'return_pct': tr*100, 'pnl_usd': pnl,
              'cumulative_pnl': np.cumsum(pnl), 'is_winner': tr > 0
}).to_csv(os.path.join(LATEST_DIR, "trades.csv"), index=False)
print(f"\u2705 trades.csv")

results_df.to_csv(os.path.join(LATEST_DIR, "grid_results.csv"), index=False)
print(f"\u2705 grid_results.csv")

# Run log
log_path = os.path.join(EXPORT_DIR, "run_log.csv")
log_entry = pd.DataFrame([{
    "run_id": RUN_ID, "timestamp": RUN_TIMESTAMP.isoformat(), "strategy": STRATEGY_NAME,
    "ticker": TICKER, "best_params": str(best_params),
    "sharpe_full": round(M['sharpe'] or 0, 4), "sharpe_is": round(M['is_sharpe'] or 0, 4),
    "sharpe_oos": round(M['oos_sharpe'] or 0, 4), "total_return": round(M['total_return'] or 0, 4),
    "max_drawdown": round(M['max_dd'] or 0, 4), "total_trades": M['trades'],
    "win_rate": round(M['win_rate'] or 0, 1), "profit_factor": round(M['pf'] or 0, 2) if M['pf'] else None,
    "notes": NOTES, "export_path": STRAT_DIR,
}])
if os.path.exists(log_path):
    log_combined = pd.concat([pd.read_csv(log_path), log_entry], ignore_index=True)
else:
    log_combined = log_entry
log_combined.to_csv(log_path, index=False)
print(f"\u2705 run_log.csv ({len(log_combined)} runs)")

# ════════════════════════════════════════════════════════════════
# 2. GENERATE PDF TEARSHEET — Clean White Card Design
# ════════════════════════════════════════════════════════════════
pdf_path = os.path.join(LATEST_DIR, "tearsheet.pdf")
pdf_archive = os.path.join(ARCHIVE_DIR, f"{RUN_ID}_tearsheet.pdf")

fmt = lambda v, d=2: f"{v:.{d}f}" if v is not None and not np.isnan(v) else "N/A"
fmtp = lambda v: f"{v:.2%}" if v is not None and not np.isnan(v) else "N/A"

# ── Design tokens ──
BG       = '#FFFFFF'
CARD_BG  = '#F7F8FA'
CARD_BRD = '#E2E5EB'
TEXT_PRI = '#1A1D23'
TEXT_SEC = '#5A6270'
TEXT_MUT = '#8C95A3'
ACCENT   = '#2563EB'   # Blue
GREEN    = '#059669'
RED      = '#DC2626'
ORANGE   = '#EA580C'
GRID_CLR = '#E5E7EB'

def _draw_card(ax_fig, x, y, w, h, label, value, color=ACCENT, fontsize_val=22):
    """Draw a rounded metric card on the figure."""
    from matplotlib.patches import FancyBboxPatch
    rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.008",
                           facecolor=CARD_BG, edgecolor=CARD_BRD, linewidth=1.2,
                           transform=ax_fig.transAxes, zorder=2)
    ax_fig.add_patch(rect)
    ax_fig.text(x + w/2, y + h*0.68, value, ha='center', va='center',
                fontsize=fontsize_val, fontweight='bold', color=color,
                transform=ax_fig.transAxes, zorder=3)
    ax_fig.text(x + w/2, y + h*0.25, label, ha='center', va='center',
                fontsize=8, color=TEXT_SEC, transform=ax_fig.transAxes, zorder=3)

def _style_ax(ax, title=None):
    """Apply clean white styling to an axis."""
    ax.set_facecolor(BG)
    ax.tick_params(colors=TEXT_SEC, labelsize=8)
    ax.grid(True, alpha=0.4, color=GRID_CLR, linewidth=0.8)
    for spine in ax.spines.values():
        spine.set_color(CARD_BRD)
        spine.set_linewidth(0.8)
    if title:
        ax.set_title(title, color=TEXT_PRI, fontsize=11, fontweight='bold', pad=10)

with PdfPages(pdf_path) as pdf:

    # ── PAGE 1: Dashboard with Metric Cards + Summary Table ──
    fig = plt.figure(figsize=(11, 8.5))
    fig.patch.set_facecolor(BG)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

    # Header bar
    from matplotlib.patches import FancyBboxPatch, Rectangle
    header = Rectangle((0, 0.91), 1, 0.09, facecolor=ACCENT, transform=ax.transAxes, zorder=1)
    ax.add_patch(header)
    ax.text(0.04, 0.955, "STRATEGY TEARSHEET", ha='left', va='center', fontsize=20,
            fontweight='bold', color='white', transform=ax.transAxes, zorder=2)
    ax.text(0.96, 0.955, f"{STRATEGY_NAME}  |  {TICKER}", ha='right', va='center',
            fontsize=12, color='rgba(255,255,255,0.85)', transform=ax.transAxes, zorder=2, family='monospace')
    ax.text(0.96, 0.92, f"{full_close.index[0].date()} to {full_close.index[-1].date()}  |  {len(full_close)} bars  |  {param_str}",
            ha='right', va='center', fontsize=8, color='rgba(255,255,255,0.65)', transform=ax.transAxes, zorder=2)

    # Metric cards row — top KPIs
    card_w, card_h = 0.145, 0.09
    card_y = 0.79
    cards = [
        ("SHARPE", fmt(M['sharpe'], 3), ACCENT),
        ("RETURN", fmtp(M['total_return']), GREEN if (M['total_return'] or 0) > 0 else RED),
        ("MAX DD", fmtp(M['max_dd']), RED if (M['max_dd'] or 0) < -0.15 else ORANGE),
        ("WIN RATE", f"{M['win_rate']:.1f}%" if M['win_rate'] else "N/A", GREEN if (M['win_rate'] or 0) > 50 else RED),
        ("TRADES", str(M['trades']), TEXT_PRI),
        ("PROFIT FACTOR", fmt(M['pf']), GREEN if (M['pf'] or 0) > 1.5 else (ORANGE if (M['pf'] or 0) > 1 else RED)),
    ]
    x_start = 0.03
    gap = (0.94 - len(cards) * card_w) / (len(cards) - 1)
    for i, (label, value, color) in enumerate(cards):
        cx = x_start + i * (card_w + gap)
        _draw_card(ax, cx, card_y, card_w, card_h, label, value, color)

    # IS vs OOS comparison table
    table_y = 0.04
    table_h = 0.70
    rows = [
        ["METRIC", "FULL SAMPLE", "IN-SAMPLE", "OUT-OF-SAMPLE", "BUY & HOLD"],
        ["Total Return", fmtp(M['total_return']), fmtp(M['is_return']), fmtp(M['oos_return']), fmtp(M['bh_return'])],
        ["Sharpe Ratio", fmt(M['sharpe'], 3), fmt(M['is_sharpe'], 3), fmt(M['oos_sharpe'], 3), fmt(M['bh_sharpe'], 3)],
        ["Sortino Ratio", fmt(M['sortino'], 3), "—", "—", "—"],
        ["Max Drawdown", fmtp(M['max_dd']), fmtp(M['is_dd']), fmtp(M['oos_dd']), fmtp(M['bh_dd'])],
        ["Calmar Ratio", fmt(M['calmar'], 3), "—", "—", "—"],
        ["Volatility (Ann.)", fmtp(M['volatility']), "—", "—", "—"],
        ["Win Rate", f"{M['win_rate']:.1f}%" if M['win_rate'] else "N/A", "—", "—", "—"],
        ["Profit Factor", fmt(M['pf']), "—", "—", "—"],
        ["Expectancy", fmt(M['expectancy'], 4), "—", "—", "—"],
        ["Payoff Ratio", fmt(M['payoff']), "—", "—", "—"],
        ["Total Trades", str(M['trades']), str(M['is_trades']), str(M['oos_trades']), "1"],
        ["Trades/Year", fmt(M['trades_yr'], 1), "—", "—", "—"],
        ["Avg Win", fmtp(M['avg_win']), "—", "—", "—"],
        ["Avg Loss", fmtp(M['avg_loss']), "—", "—", "—"],
        ["Largest Win", fmtp(M['largest_win']), "—", "—", "—"],
        ["Largest Loss", fmtp(M['largest_loss']), "—", "—", "—"],
    ]

    table = ax.table(cellText=rows[1:], colLabels=rows[0], cellLoc='center', loc='center',
                     bbox=[0.03, table_y, 0.94, table_h])
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor(CARD_BRD)
        cell.set_linewidth(0.6)
        if row == 0:
            cell.set_facecolor(ACCENT)
            cell.set_text_props(color='white', fontweight='bold', fontsize=8)
        else:
            cell.set_facecolor(BG if row % 2 == 1 else CARD_BG)
            cell.set_text_props(color=TEXT_PRI, fontsize=8, family='monospace')
            if col == 0:
                cell.set_text_props(color=TEXT_PRI, fontsize=8, fontweight='bold')
        cell.set_height(0.052)

    # Footer
    ax.text(0.5, 0.01, f"Run {RUN_ID}  |  QS Finance", ha='center', va='bottom',
            fontsize=7, color=TEXT_MUT, transform=ax.transAxes)

    pdf.savefig(fig, facecolor=BG)
    plt.close(fig)

    # ── PAGE 2: Equity Curve + Drawdown ──
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8.5), gridspec_kw={'height_ratios': [3, 1]})
    fig.patch.set_facecolor(BG)
    fig.suptitle(f'{STRATEGY_NAME} on {TICKER} — Equity & Drawdown', fontsize=14,
                 fontweight='bold', color=TEXT_PRI, y=0.97)

    eq_strat = pf_full.value(); eq_bh = pf_bh.value()

    # Equity curve with gradient fill
    _style_ax(ax1)
    ax1.plot(full_close.index[:split_idx], eq_strat.iloc[:split_idx].values,
             color=ACCENT, linewidth=2, label='Strategy (IS)', solid_capstyle='round')
    ax1.plot(full_close.index[split_idx:], eq_strat.iloc[split_idx:].values,
             color=ORANGE, linewidth=2, label='Strategy (OOS)', solid_capstyle='round')
    ax1.plot(full_close.index, eq_bh.values, color=TEXT_MUT, linewidth=1.2,
             alpha=0.6, linestyle='--', label='Buy & Hold')
    ax1.axvline(x=full_close.index[split_idx], color=RED, linestyle=':', alpha=0.3, linewidth=1)
    ax1.fill_between(full_close.index[:split_idx], eq_strat.iloc[:split_idx].values,
                      eq_strat.iloc[:split_idx].values.min(), alpha=0.06, color=ACCENT)
    ax1.fill_between(full_close.index[split_idx:], eq_strat.iloc[split_idx:].values,
                      eq_strat.iloc[split_idx:].values.min(), alpha=0.06, color=ORANGE)
    ax1.set_ylabel('Portfolio Value ($)', color=TEXT_SEC, fontsize=9)
    ax1.legend(fontsize=8, facecolor=BG, edgecolor=CARD_BRD, labelcolor=TEXT_PRI, framealpha=0.95)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    # Stats banner
    stats_text = f"Sharpe: {fmt(M['sharpe'],3)}   |   Return: {fmtp(M['total_return'])}   |   MaxDD: {fmtp(M['max_dd'])}   |   WR: {M['win_rate']:.1f}%   |   PF: {fmt(M['pf'])}"
    ax1.text(0.5, 0.03, stats_text, transform=ax1.transAxes, fontsize=8, ha='center',
             color=TEXT_SEC, family='monospace',
             bbox=dict(boxstyle='round,pad=0.5', facecolor=CARD_BG, edgecolor=CARD_BRD, alpha=0.95))

    # Drawdown
    _style_ax(ax2)
    dd = pf_full.drawdown() * 100
    ax2.fill_between(full_close.index, dd.values, 0, color=RED, alpha=0.2)
    ax2.plot(full_close.index, dd.values, color=RED, linewidth=0.8, alpha=0.7)
    ax2.set_ylabel('Drawdown %', color=TEXT_SEC, fontsize=9)
    ax2.set_xlabel('Date', color=TEXT_SEC, fontsize=9)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    pdf.savefig(fig, facecolor=BG)
    plt.close(fig)

    # ── PAGE 3: Trade Analysis — 2x2 Grid ──
    if len(tr) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))
        fig.patch.set_facecolor(BG)
        fig.suptitle(f'Trade-by-Trade Analysis — {len(tr)} Trades', fontsize=14,
                     fontweight='bold', color=TEXT_PRI, y=0.97)

        n = len(tr)
        colors_bar = [GREEN if r > 0 else RED for r in tr]
        colors_pnl = [GREEN if p > 0 else RED for p in pnl]

        for a in axes.flat:
            _style_ax(a)

        # Trade returns bar chart
        axes[0,0].bar(range(n), tr*100, color=colors_bar, edgecolor='none', width=0.8, alpha=0.85)
        axes[0,0].axhline(np.mean(tr)*100, color=ACCENT, linestyle='--', linewidth=1.5, label=f'Avg: {np.mean(tr)*100:.2f}%')
        axes[0,0].axhline(0, color=TEXT_MUT, linewidth=0.5)
        axes[0,0].set_title('Trade Returns (%)', color=TEXT_PRI, fontsize=11, fontweight='bold')
        axes[0,0].set_xlabel('Trade #', color=TEXT_SEC, fontsize=8)
        axes[0,0].legend(fontsize=7, facecolor=BG, edgecolor=CARD_BRD)

        # Trade P&L bar chart
        axes[0,1].bar(range(n), pnl, color=colors_pnl, edgecolor='none', width=0.8, alpha=0.85)
        axes[0,1].axhline(np.mean(pnl), color=ACCENT, linestyle='--', linewidth=1.5, label=f'Avg: ${np.mean(pnl):,.0f}')
        axes[0,1].axhline(0, color=TEXT_MUT, linewidth=0.5)
        axes[0,1].set_title('Trade P&L ($)', color=TEXT_PRI, fontsize=11, fontweight='bold')
        axes[0,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        axes[0,1].legend(fontsize=7, facecolor=BG, edgecolor=CARD_BRD)

        # Cumulative P&L with gradient
        cum_pnl = np.cumsum(pnl)
        axes[1,0].plot(range(1, n+1), cum_pnl, color=ACCENT, linewidth=2.5, solid_capstyle='round')
        axes[1,0].fill_between(range(1, n+1), cum_pnl, 0, where=cum_pnl>=0, alpha=0.12, color=GREEN)
        axes[1,0].fill_between(range(1, n+1), cum_pnl, 0, where=cum_pnl<0, alpha=0.12, color=RED)
        axes[1,0].axhline(0, color=TEXT_MUT, linewidth=0.5)
        axes[1,0].set_title('Cumulative P&L', color=TEXT_PRI, fontsize=11, fontweight='bold')
        axes[1,0].set_xlabel('Trade #', color=TEXT_SEC, fontsize=8)
        axes[1,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

        # Return distribution histogram
        axes[1,1].hist(tr*100, bins=min(30, max(10, n//3)), color=ACCENT, edgecolor='white',
                       alpha=0.75, linewidth=0.5)
        axes[1,1].axvline(np.mean(tr)*100, color=RED, linestyle='--', linewidth=2, label=f'Mean: {np.mean(tr)*100:.2f}%')
        axes[1,1].axvline(0, color=TEXT_MUT, linewidth=0.8, alpha=0.5)
        axes[1,1].set_title('Return Distribution', color=TEXT_PRI, fontsize=11, fontweight='bold')
        axes[1,1].set_xlabel('Return %', color=TEXT_SEC, fontsize=8)
        axes[1,1].legend(fontsize=7, facecolor=BG, edgecolor=CARD_BRD)

        plt.tight_layout(rect=[0, 0, 1, 0.95])
        pdf.savefig(fig, facecolor=BG)
        plt.close(fig)

    # ── PAGE 4: Monte Carlo FTMO Simulation ──
    dr = daily_rets.values.ravel(); dr = dr[~np.isnan(dr)]
    N_SIM = 5000; DAYS = 30; ACCOUNT = 100000
    np.random.seed(42)
    n_passed = n_blown_t = n_blown_d = 0
    final_eqs = np.zeros(N_SIM); sample_paths = []

    for s in range(N_SIM):
        eq = ACCOUNT; path = [ACCOUNT]
        sim_rets = np.random.choice(dr, size=DAYS, replace=True)
        blown = False
        for d in range(DAYS):
            day_start = eq; eq *= (1 + sim_rets[d]); path.append(eq)
            if (eq - day_start)/ACCOUNT < -0.05: n_blown_d += 1; blown = True; break
            if (eq - ACCOUNT)/ACCOUNT < -0.10: n_blown_t += 1; blown = True; break
            if (eq - ACCOUNT)/ACCOUNT >= 0.10: n_passed += 1; blown = True; break
        while len(path) < DAYS + 1: path.append(path[-1])
        final_eqs[s] = path[-1]
        if s < 150: sample_paths.append(path)

    n_still = N_SIM - n_passed - n_blown_t - n_blown_d
    pass_rate = n_passed / N_SIM * 100

    verdict = "FAVORABLE" if pass_rate >= 50 else "POSSIBLE" if pass_rate >= 25 else "CHALLENGING" if pass_rate >= 10 else "UNLIKELY"
    verdict_color = GREEN if pass_rate >= 50 else ORANGE if pass_rate >= 25 else (ORANGE if pass_rate >= 10 else RED)

    fig = plt.figure(figsize=(11, 8.5))
    fig.patch.set_facecolor(BG)

    # Top section: MC result cards
    ax_top = fig.add_axes([0, 0.82, 1, 0.18])
    ax_top.set_xlim(0, 1); ax_top.set_ylim(0, 1); ax_top.axis('off')

    # Title
    ax_top.text(0.5, 0.85, f'FTMO Monte Carlo — {N_SIM:,} Simulations', ha='center', va='center',
                fontsize=16, fontweight='bold', color=TEXT_PRI, transform=ax_top.transAxes)

    # Result cards
    mc_cards = [
        ("PASS RATE", f"{pass_rate:.1f}%", verdict_color),
        ("VERDICT", verdict, verdict_color),
        ("PASSED", f"{n_passed:,}", GREEN),
        ("BLOWN (TOTAL)", f"{n_blown_t:,}", RED),
        ("BLOWN (DAILY)", f"{n_blown_d:,}", RED),
        ("STILL TRADING", f"{n_still:,}", TEXT_SEC),
    ]
    mc_cw = 0.14
    mc_gap = (0.92 - len(mc_cards) * mc_cw) / (len(mc_cards) - 1)
    for i, (label, value, color) in enumerate(mc_cards):
        cx = 0.04 + i * (mc_cw + mc_gap)
        _draw_card(ax_top, cx, 0.05, mc_cw, 0.65, label, value, color, fontsize_val=16)

    # Bottom section: Equity paths
    ax_mc = fig.add_axes([0.08, 0.08, 0.86, 0.70])
    _style_ax(ax_mc)

    for path in sample_paths:
        c = GREEN if path[-1] >= 110000 else (RED if path[-1] <= 90000 else TEXT_MUT)
        ax_mc.plot(range(DAYS+1), path, color=c, alpha=0.15, linewidth=0.5)
    ax_mc.axhline(110000, color=GREEN, linestyle='--', linewidth=2.5, label=f'10% Target ($110k)')
    ax_mc.axhline(90000, color=RED, linestyle='--', linewidth=2.5, label=f'10% Max Loss ($90k)')
    ax_mc.axhline(100000, color=TEXT_MUT, linestyle='-', linewidth=0.8, alpha=0.4)

    ax_mc.set_xlabel('Trading Day', color=TEXT_SEC, fontsize=9)
    ax_mc.set_ylabel('Equity ($)', color=TEXT_SEC, fontsize=9)
    ax_mc.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax_mc.legend(fontsize=9, facecolor=BG, edgecolor=CARD_BRD, labelcolor=TEXT_PRI, framealpha=0.95)

    # Median final equity annotation
    ax_mc.text(0.5, 0.03, f"Median Final Equity: ${np.median(final_eqs):,.0f}", transform=ax_mc.transAxes,
               fontsize=9, ha='center', color=TEXT_SEC, family='monospace',
               bbox=dict(boxstyle='round,pad=0.5', facecolor=CARD_BG, edgecolor=CARD_BRD, alpha=0.95))

    pdf.savefig(fig, facecolor=BG)
    plt.close(fig)

# Copy PDF to archive
import shutil
shutil.copy2(pdf_path, pdf_archive)

# Add MC results to JSON
mc_data = {"pass_rate": round(pass_rate, 1), "n_simulations": N_SIM, "n_passed": n_passed,
           "n_blown_total": n_blown_t, "n_blown_daily": n_blown_d, "n_still_trading": n_still,
           "median_final_equity": round(float(np.median(final_eqs)), 2), "verdict": verdict}
export_json["monte_carlo_ftmo"] = mc_data
with open(os.path.join(LATEST_DIR, "summary.json"), 'w') as f:
    json.dump(export_json, f, indent=2, default=str)


# ════════════════════════════════════════════════════════════════
# LOG TO GOOGLE SHEETS
# ════════════════════════════════════════════════════════════════
try:
    from lib.sheets_logger import SheetsLogger
    _logger = SheetsLogger()
    _logger.log_result(export_json)
    # Log trades if available
    _trades_path = os.path.join(LATEST_DIR, "trades.csv")
    if os.path.exists(_trades_path):
        _trades_df = pd.read_csv(_trades_path)
        _logger.log_trades(TICKER, STRATEGY_NAME, RUN_ID, _trades_df)
except Exception as _e:
    print(f"⚠️ Sheets logging skipped: {_e}")

# ════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ════════════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print(f"\U0001f4e6 EXPORT COMPLETE — {STRATEGY_NAME} on {TICKER}")
print(f"{'='*70}")
print(f"  Run ID:       {RUN_ID}")
print(f"  Timestamp:    {RUN_TIMESTAMP.strftime('%B %d, %Y at %I:%M:%S %p')}")
print(f"  Instrument:   {TICKER} ({export_json['metadata']['instrument_type']})")
print(f"  Params:       {param_str}")
print(f"  Sharpe:       {fmt(M['sharpe'],3)} (IS: {fmt(M['is_sharpe'],3)} -> OOS: {fmt(M['oos_sharpe'],3)})")
print(f"  FTMO Verdict: {verdict} ({pass_rate:.1f}% pass rate)")
print(f"{'='*70}")
print(f"\n\U0001f4c2 {STRAT_DIR}/")
print(f"  \u251c\u2500\u2500 latest/")
print(f"  \u2502   \u251c\u2500\u2500 summary.json")
print(f"  \u2502   \u251c\u2500\u2500 daily_returns.csv")
print(f"  \u2502   \u251c\u2500\u2500 trades.csv")
print(f"  \u2502   \u251c\u2500\u2500 grid_results.csv")
print(f"  \u2502   \u2514\u2500\u2500 tearsheet.pdf       \u2190 Share this with Claude!")
print(f"  \u2514\u2500\u2500 archive/")
print(f"      \u251c\u2500\u2500 {RUN_ID}_summary.json")
print(f"      \u2514\u2500\u2500 {RUN_ID}_tearsheet.pdf")
print(f"\n\U0001f4cb run_log.csv ({len(log_combined)} total runs)")
print(f"\n\U0001f4a1 To get my analysis: upload the tearsheet.pdf to our chat.")
print(f"   For deeper analysis: also upload summary.json + daily_returns.csv")

